In [ ]:
# 1. Install required packages (100% Offline Mode - including z3-solver)
import os
import sys
import subprocess
import glob

os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

offline_pkg_dir = "/kaggle/input/datasets/mduy2911/offline-packages"
print(f"Installing offline packages from: {offline_pkg_dir}...")
wheels = glob.glob(os.path.join(offline_pkg_dir, "*.whl"))

if wheels:
    cmd = [sys.executable, "-m", "pip", "install", "--no-index", "--no-deps"] + wheels
    subprocess.run(cmd, check=True)
    print("Offline installation completed successfully!")
else:
    raise FileNotFoundError(f"No offline wheels found in {offline_pkg_dir}!")

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
# 2. TRAINING HYPERPARAMETERS (Optimized for RTX Pro 6000 Ada / 96GB VRAM)
import os

MODEL_ID = "/kaggle/input/datasets/mduy2911/model-cache"

# --- RTX Pro 6000 Hardware Optimizations ---
USE_QLORA = False
GRADIENT_CHECKPOINTING = True

# --- Training Hyperparameters ---
MAX_LENGTH = 768            # Maximum sequence length
BATCH_SIZE = 16            # Physical batch size optimized for RTX 6000
GRADIENT_ACCUMULATION = 2  # Gradient accumulation steps (Effective batch size = 32)
EPOCHS = 2                  # Number of training epochs per run
LEARNING_RATE = 1e-4        # Learning rate
OUTPUT_DIR_AUG = "/kaggle/working/results_augmented"          # Output directory for Augmented Run
OUTPUT_DIR_NO_AUG = "/kaggle/working/results_no_augmentation"  # Output directory for No-Augmentation Run

os.environ["WANDB_MODE"] = "disabled"
USE_WANDB = False


In [ ]:
# 3. Load NL -> FOL Translation Datasets (Paths left blank per user request)
import os
import json

merged_path = "/kaggle/input/datasets/mduy2911/folc-train/logic_merged_valid_augmented.json"
no_aug_path = "/kaggle/input/datasets/mduy2911/folc-train/logic_merged_valid.json" 

def load_translation_dataset(path):
    if not path or not os.path.exists(path):
        print(f"Warning: Path '{path}' is empty or does not exist. Skipping load.")
        return []
    samples = []
    seen_premises = set()
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    for item in data:
        nl_list = item.get("premises-NL", [])
        fol_list = item.get("premises-FOL", [])
        if not nl_list or not fol_list or len(nl_list) != len(fol_list):
            continue
        nl_serialized = "\n".join(nl_list)
        if nl_serialized in seen_premises:
            continue
        seen_premises.add(nl_serialized)
        
        sample_dict = {"premises-NL": nl_list, "premises-FOL": fol_list}
        if "split" in item:
            sample_dict["split"] = item["split"]
        samples.append(sample_dict)
        
    print(f"Loaded {len(samples)} unique translation samples from {os.path.basename(path)}")
    return samples

raw_samples_aug = load_translation_dataset(merged_path)
raw_samples_no_aug = load_translation_dataset(no_aug_path)


In [ ]:
# 4. Initialize Tokenizer and Load Helper Functions
import os
import sys
import gc
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, PeftModel

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Check hardware bfloat16 support
if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
    compute_dtype = torch.bfloat16
    use_bf16 = True
    use_fp16 = False
    print("GPU supports bfloat16. Using bfloat16 compute (Optimal for Ampere/Ada/Hopper GPUs like RTX Pro 6000).")
else:
    compute_dtype = torch.float16
    use_bf16 = False
    use_fp16 = True
    print("Using float16 compute (Standard for Turing/Pascal/Volta GPUs or CPU).")

# Select the most optimal Attention implementation
attn_impl = "sdpa"
try:
    import flash_attn
    attn_impl = "flash_attention_2"
    print("FlashAttention-2 is installed. Using flash_attention_2.")
except ImportError:
    print("FlashAttention-2 not found. Using PyTorch Native SDPA (Scaled Dot Product Attention).")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID, 
    trust_remote_code=True, 
    local_files_only=True
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def load_base_model():
    print("Loading a fresh instance of the base model...")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        dtype=compute_dtype if torch.cuda.is_available() else torch.float32,
        device_map="auto" if torch.cuda.is_available() else None,
        trust_remote_code=True,
        attn_implementation=attn_impl,
        local_files_only=True
)
    model.config.use_cache = False
    return model

def clean_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("GPU and system memory cleaned.")

target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

# Optimized LoRA Configuration (Rank = 32, Alpha = 64) for logical capacity
peft_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=target_modules,
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)


In [ ]:
# 5. Helper Function to Format Dataset (Chat Template) and split Train/Val
import json
import re
import random
from datasets import Dataset

# Define prompt templates for flat JSON list output with strict count constraint
system_prompt_template = (
    "You convert natural-language premises into parser-safe first-order logic formulas.\n\n"
    "Output a STRICT valid JSON list of strings containing the first-order logic formulas in the exact order of the input premises.\n"
    "You must output EXACTLY the same number of formulas as the input premises. Do not skip any premises or merge them.\n\n"
    "ALLOWED OPERATORS:\n"
    "AND, OR, NOT, ->, <->, =, !=, >=, <=, >, <, ForAll, Exists\n\n"
    "QUANTIFIER RULES:\n"
    "Use nested quantifiers only. E.g., ForAll(x, ForAll(y, P(x,y)))\n\n"
    "Return JSON only."
)

user_prompt_template = (
    "Convert the following {num_premises} premises into canonical first-order logic.\n\n"
    "Premises:\n"
    "{premises}\n\n"
    "Return a JSON list of exactly {num_premises} strings containing the formulas, in the exact same order."
)

def normalize_fol(formula):
    # Standardize spaces and keep logic keywords
    keywords = {"ForAll", "Exists", "AND", "OR", "NOT", "In", "->", "<->", "=", "!=", ">=", "<=", ">", "<"}
    # Extract all word-like identifiers
    words = re.findall(r"\b[A-Za-z_][A-Za-z0-9_-]*\b", formula)
    word_map = {}
    counter = 0
    normalized = formula
    for w in words:
        if w in keywords or w.isdigit() or w.startswith("VAR_"):
            continue
        if w not in word_map:
            word_map[w] = f"VAR_{counter}"
            counter += 1
        normalized = re.sub(r"\b" + re.escape(w) + r"\b", word_map[w], normalized)
    return normalized

def get_structure_key(sample):
    fol_list = sample.get("premises-FOL", [])
    normalized_list = [normalize_fol(f) for f in fol_list]
    return "||".join(normalized_list)

def format_hf_dataset_only(raw_samples, tokenizer):
    if not raw_samples:
        return None
    
    formatted_samples = []
    for item in raw_samples:
        nl_list = item["premises-NL"]
        fol_list = item["premises-FOL"]
        
        # Format premises as a numbered list
        nl_content = ""
        for i, nl in enumerate(nl_list, start=1):
            nl_content += f"{i}. {nl}\n"
            
        # Render user prompt template with count constraint
        user_prompt = user_prompt_template.format(num_premises=len(nl_list), premises=nl_content.strip())
        
        # Assistant response is a flat JSON list of FOL formulas
        assistant_response = json.dumps(fol_list, indent=2)
        
        formatted_samples.append({
            "messages": [
                {"role": "system", "content": system_prompt_template},
                {"role": "user", "content": user_prompt.strip()},
                {"role": "assistant", "content": assistant_response.strip()}
            ]
        })

    dataset = Dataset.from_list(formatted_samples)

    def apply_template(batch):
        texts = []
        for messages in batch["messages"]:
            text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
            texts.append(text)
        return {"text": texts}

    dataset = dataset.map(apply_template, batched=True, remove_columns=["messages"])
    return dataset

def prepare_hf_dataset(raw_samples, tokenizer, test_size=0.1, seed=42):
    if not raw_samples:
        return None, None
        
    # Check if pre-split is defined in the dataset
    has_presplit = all("split" in sample for sample in raw_samples)
    if has_presplit:
        train_samples = [s for s in raw_samples if s.get("split") == "train"]
        val_samples = [s for s in raw_samples if s.get("split") == "val"]
        print(f"Loaded pre-split dataset - Train: {len(train_samples)}, Val: {len(val_samples)}")
    else:
        # Group samples by their structure key to prevent structural leakage
        groups = {}
        for sample in raw_samples:
            key = get_structure_key(sample)
            if key not in groups:
                groups[key] = []
            groups[key].append(sample)
            
        # Sort keys to be 100% deterministic before shuffling
        unique_keys = sorted(list(groups.keys()))
        
        # Shuffle keys deterministically using a local random generator with a fixed seed
        rng = random.Random(seed)
        rng.shuffle(unique_keys)
        
        # Split keys
        split_idx = int(len(unique_keys) * (1 - test_size))
        train_keys = set(unique_keys[:split_idx])
        val_keys = set(unique_keys[split_idx:])
        
        train_samples = []
        val_samples = []
        for key in unique_keys:
            if key in val_keys:
                val_samples.extend(groups[key])
            else:
                train_samples.extend(groups[key])
            
    # Format to Hugging Face datasets
    train_dataset = format_hf_dataset_only(train_samples, tokenizer)
    val_dataset = format_hf_dataset_only(val_samples, tokenizer)
    
    print(f"Final HF Dataset size - Train: {len(train_dataset)}, Val: {len(val_dataset)}")
    return train_dataset, val_dataset


In [ ]:
# 6. SFT Training Setup and Callbacks
import os
import glob
import torch
from trl import SFTTrainer, SFTConfig
from transformers import TrainerCallback, DataCollatorForLanguageModeling
from typing import Dict, Union, Any, Optional, List, Tuple

class LossLoggingCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is not None:
            if "loss" in logs:
                print(f"Step {state.global_step}: training_loss = {logs['loss']:.6f}")
            if "eval_loss" in logs:
                print(f"Step {state.global_step}: validation_loss = {logs['eval_loss']:.6f}")

class CustomDataCollator(DataCollatorForLanguageModeling):
    def __init__(self, response_template, tokenizer, *args, **kwargs):
        super().__init__(tokenizer=tokenizer, mlm=False, *args, **kwargs)
        self.response_template = response_template
        self.response_token_ids = self.tokenizer.encode(self.response_template, add_special_tokens=False)
        
    def __call__(self, examples):
        batch = super().__call__(examples)
        labels = batch["labels"]
        for i in range(labels.size(0)):
            input_ids = batch["input_ids"][i].tolist()
            response_idx = -1
            n_template = len(self.response_token_ids)
            for j in range(len(input_ids) - n_template + 1):
                if input_ids[j:j+n_template] == self.response_token_ids:
                    response_idx = j + n_template
                    break
            
            if response_idx != -1:
                labels[i, :response_idx] = -100
                
        return batch

def train_model(train_dataset, val_dataset, output_dir):
    clean_memory()
    
    # Mathematically exact, dynamic warmup steps calculation based on actual dataset size
    num_samples = len(train_dataset)
    effective_batch_size = BATCH_SIZE * GRADIENT_ACCUMULATION
    steps_per_epoch = num_samples // effective_batch_size
    if num_samples % effective_batch_size != 0:
        steps_per_epoch += 1
    total_steps = steps_per_epoch * EPOCHS
    
    # Target exactly 3% warmup steps of total training steps (HF v5.2 compliant integer steps)
    warmup_steps = max(1, int(total_steps * 0.05))
    print(f"Training on {num_samples} samples. Steps per epoch: {steps_per_epoch}. Total steps: {total_steps}. Dynamic warmup steps: {warmup_steps}")
    
    base_model = load_base_model()
    
    print("Initializing a new PEFT adapter...")
    model = get_peft_model(base_model, peft_config)
    model.print_trainable_parameters()
    
    # Highly Optimized SFTConfig with Cosine Decay Scheduler & Warmup
    sft_config = SFTConfig(
        output_dir=output_dir,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION,
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=LEARNING_RATE,
        lr_scheduler_type="cosine",             # Cosine learning rate decay scheduler
        warmup_steps=warmup_steps,              # Compliant, dynamic warm-up steps to stabilize updates
        fp16=use_fp16,
        bf16=use_bf16,
        logging_steps=1,
        logging_first_step=True,
        optim="adamw_torch",
        gradient_checkpointing=GRADIENT_CHECKPOINTING,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        per_device_eval_batch_size=BATCH_SIZE,
        report_to="none",
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        save_total_limit=2,
        dataset_text_field="text",
        max_length=MAX_LENGTH,
        neftune_noise_alpha=None,
        dataloader_num_workers=2,
        dataloader_pin_memory=True
    )
    
    response_template = "<|im_start|>assistant\n"
    collator = CustomDataCollator(response_template=response_template, tokenizer=tokenizer)
    
    trainer = SFTTrainer(
        model=model,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        processing_class=tokenizer,
        args=sft_config,
        data_collator=collator,
        callbacks=[LossLoggingCallback()]
    )
    
    checkpoints = glob.glob(os.path.join(output_dir, "checkpoint-*"))
    resume_path = None
    if checkpoints:
        checkpoints.sort(key=lambda x: int(x.split("-")[-1]))
        resume_path = checkpoints[-1]
        print(f"Found checkpoints. Resuming training from: {resume_path}")
        
    trainer.train(resume_from_checkpoint=resume_path)
    
    print(f"Saving best validation adapter weights and tokenizer to {output_dir}...")
    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)
    print("Training completed successfully!")
    
    # Clean up model & trainer from memory
    del trainer
    del model
    del base_model
    clean_memory()


In [ ]:
# 6.1 Run Training: Augmented Dataset (Run 1)
if not raw_samples_aug:
    print("Skipping Run 1 training because raw_samples_aug is empty. Please set merged_path.")
else:
    print("=== Starting Training Run 1: Augmented Dataset ===")
    train_dataset_aug, val_dataset_aug = prepare_hf_dataset(raw_samples_aug, tokenizer)
    train_model(train_dataset_aug, val_dataset_aug, OUTPUT_DIR_AUG)
